In [5]:
import sys
import os
import pandas as pd
import numpy as np

src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

from config import SIM_CONFIG
from users import generate_users
from sessions import generate_sessions
from events import generate_events
from orders import generate_orders
from order_items import generate_order_items

rng = np.random.default_rng(seed=42)
print("Modules loaded successfully.")

Modules loaded successfully.


In [6]:
test_users_df = generate_users(total_users=1000000, rng=rng)
display(test_users_df.head())

Generating 1000000 users
users table generated!


,user_id,account_created_at,city,region,user_tenure_days,latent_income_score,latent_digital_literacy,latent_trust_in_platform
0,265bb85784a8457ab5e51465d5a81cc2,2024-04-16 05:42:14,south_cotabato,region_xii,258,0.411423,0.728934,0.225373
1,856c969017324c0cb5c1858a8b3221f0,2024-11-09 02:06:16,ilocos_norte,region_i,51,0.197514,0.064570,0.510819
2,5732fbec2d3649d99da284605a338a73,2023-10-16 14:03:19,sorsogon,region_v,441,0.417296,0.014083,0.672436
3,7529a07af1b44bb4b1b7edd34549f537,2023-06-25 12:09:40,ilocos_norte,region_i,554,0.687345,0.214870,0.426581
4,3228a2e46179490aaadb7bbf09748b37,2024-12-13 09:02:37,davao_del_sur,region_xi,17,3.017209,0.134020,0.294636


In [7]:
test_sessions_df = generate_sessions(df_users=test_users_df, rng=rng)
display(test_sessions_df.head())

Generating 1900225 sessions
sessions table generated!


,session_id,user_id,session_number,acq_channel,utm_source,utm_medium,utm_campaign,session_start_time,session_end_time,session_duration_seconds,device_operating_system,device_operating_system_version,device_group
0,42d496b656ef4a1b8db39cf934679491,265bb85784a8457ab5e51465d5a81cc2,1,meta,ig,cpc,retargeting,2024-04-22 14:44:44,2024-04-22 14:44:54,10,ios,16.5,mobile
1,17dfa8fd914045709f8299f5c20375f7,265bb85784a8457ab5e51465d5a81cc2,2,direct,(direct),(none),(not_set),2024-11-10 18:04:29,2024-11-10 18:10:41,372,android,14,mobile
2,f68bc030dc7e463cb19be88610521737,265bb85784a8457ab5e51465d5a81cc2,3,organic,google,organic,(not_set),2024-04-22 21:12:47,2024-04-22 21:15:06,139,windows,11,desktop
3,b6878e31674b42c5b560440650888365,856c969017324c0cb5c1858a8b3221f0,1,organic,google,organic,(not_set),2024-11-14 13:44:22,2024-11-14 13:52:58,516,ios,17.3,mobile
4,17fd369bc64c4ba6a2e825bf883e139e,856c969017324c0cb5c1858a8b3221f0,2,organic,google,organic,(not_set),2024-12-15 21:35:32,2024-12-15 21:40:32,300,android,14,mobile


In [ ]:
test_events_df = generate_events(df_sessions=test_sessions_df, df_users=test_users_df, rng=rng, n_workers=1)
display(test_events_df.head())

Generating events for 1900225 sessions


In [ ]:
test_orders_df = generate_orders(df_events=test_events_df, df_sessions=test_sessions_df, rng=rng)
display(test_orders_df.head())

In [ ]:
test_order_items_df = generate_order_items(df_orders=test_orders_df, df_users=test_users_df, rng=rng)
display(test_order_items_df.head())

In [15]:
def internal_validation(df_users, df_sessions, df_events, df_orders, df_order_items):
    print("=== BEGINNING INTERNAL DAG VALIDATION ===")

    # 1. Checking referential integrity
    purchase_sessions = df_events[df_events['event_name'] == 'purchase']['session_id'].unique()
    orphaned_orders = df_orders[~df_orders['session_id'].isin(purchase_sessions)]
    
    print("\n1. Referential Integrity:")
    if orphaned_orders.empty:
        print("   ✅ PASS: All orders have a corresponding 'purchase' event.")
    else:
        print(f"   ❌ FAIL: Found {len(orphaned_orders)} orders without a purchase event!")
        
    # 2. Funnel baseline calibration (markov chain)
    print("\n2. Markov Chain Funnel Calibration:")
    event_counts = df_events['event_name'].value_counts()
    views = event_counts.get('view_item', 1)
    purchases = event_counts.get('purchase', 0)
    conversion_rate = (purchases / views) * 100
    print(f"   Total Views: {views:,}")
    print(f"   Total Purchases: {purchases:,}")
    print(f"   View-to-Purchase Conversion Rate: {conversion_rate:.2f}% (Target: ~2.25%)")
    
    if 1.5 <= conversion_rate <= 3.0:
        print("   ✅ PASS: Conversion rate is within realistic bounds.")
    else:
        print("   ⚠️ WARNING: Conversion rate is drifting from target.")

    # 3. Behavioral Decay Assertion (Exponential)
    print(f"\n3. Temporal Decay (Digital Literacy vs. Session Duration):")
    # Merge sessions with users to access the exact column name: latent_digital_literacy
    df_sess_user = df_sessions.merge(df_users[['user_id', 'latent_digital_literacy']], on='user_id')
    
    try:
        # Calculate the deciles and the trend
        df_sess_user['literacy_decile'] = pd.qcut(df_sess_user['latent_digital_literacy'].rank(method='first'), q=10, labels=False)
        decay_trend = df_sess_user.groupby('literacy_decile')['session_duration_seconds'].mean()
        
        # Extract the exact mean durations for the lowest (0) and highest (9) deciles
        lowest_literacy_mean = decay_trend.iloc[0]
        highest_literacy_mean = decay_trend.iloc[-1]
        
        is_decreasing = lowest_literacy_mean > highest_literacy_mean
        if is_decreasing:
            print("✅ PASS: Higher digital literacy results in lower mean session duration.")
            print(f"   -> Bottom 10% Literacy (Novice): Avg {lowest_literacy_mean:.2f} seconds per session")
            print(f"   -> Top 10% Literacy (Expert): Avg {highest_literacy_mean:.2f} seconds per session")
            print(f"   -> Difference: Experts navigate {lowest_literacy_mean - highest_literacy_mean:.2f} seconds faster.")
        else:
            print("❌ FAIL: Session duration does not correlate correctly with digital literacy.")
            print(f"   -> Bottom 10% Mean: {lowest_literacy_mean:.2f}s | Top 10% Mean: {highest_literacy_mean:.2f}s")
    except Exception as e:
        print(f"⚠️ Could not calculate deciles. Error: {e}")
        
    # 4. Financial Distribution Assertion (Log-Normal)
    print("\n4. Financial Shift (Latent Income vs. Base Price):")
    df_fin = df_order_items.merge(df_orders[['order_id', 'user_id']], on='order_id')
    df_fin = df_fin.merge(df_users[['user_id', 'latent_income_score']], on='user_id')
    
    plt.figure(figsize=(8, 5))
    sns.scatterplot(data=df_fin, x='latent_income_score', y='base_item_price', alpha=0.3)
    plt.title("Internal Validation: Latent Income vs Item Price")
    plt.xlabel("Latent Income Score")
    plt.ylabel("Base Item Price (₱)")
    plt.show()

In [16]:
# Run the internal DAG and math validation
internal_validation(test_users_df, test_sessions_df, test_events_df, test_orders_df, test_order_items_df)

NameError: name 'test_users_df' is not defined